# Task 19 — GroupBy and Advanced Cricket Analytics

This task notebook is a three-dataset workout built squarely on the GroupBy machinery you mastered in the previous session. Every question is a split-apply-combine problem in disguise: split the table into meaningful groups, run an operation inside each group, and combine the answers. There are no construction exercises here — everything is analytics, and the questions wander progressively from simple grouping to genuinely hard relational cricket puzzles.

The first dataset is Titanic. Q-1 groups the passengers by Pclass and summarises the Age column inside each group — both its average and how many values are missing. Q-2 fills each class group's missing Embarked values with that group's own mode, a mode-per-group fill that a plain global fillna cannot express. Q-3 stacks two levels of grouping — Embarked first, then Pclass within each — and reports the average fare per combination, rounded to two decimals and arranged as a nested dictionary.

The second dataset is the FIFA World Cup 2022 match table. Q-4 is the heavyweight: it writes a `z_normalization(group, cols_to_perform)` function, applies it with the `apply()` method so every team's selected columns are Z-normalised — `(x - mean) / std` — and then extracts a whole battery of per-team statistics, from minimum and maximum Passes to the standard deviation of Attempted Line Breaks and the average Possession.

The third dataset is the IPL ball-by-ball log, and here the difficulty climbs steeply. Q-5 finds the batsman's highest score while chasing and the best chasing strike rate among players who have faced 100+ balls. Q-6 pits bowler against batsman — the bowler who has dismissed any one batsman the most times, with runs conceded as the tiebreaker. Q-7 identifies the most successful batting pair by total runs scored together, and Q-8 generalises it into a complete dataframe of every batting pair with Runs, Average, and Strike Rate.

Work in order: the groupby habits from the Titanic and FIFA questions are the exact machinery Q-5 to Q-8 demand. Because the tasks are long, allow repeated passes — first build each pipeline, then verify numbers against the sample outputs. When a question mentions a subtle masking rule (second innings, wides, player_out), write the mask deliberately; a small slip produces a correct-looking wrong number.

The Titanic file reads directly from a URL as with earlier tasks:

```python
import pandas as pd
url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQjh5HzZ1N0SU7ME9ZQRzeVTaXaGsV97rU8R7eAcg53k27GTstJp9cRUOfr55go1GRRvTz1NwvyOnuh/pub?gid=1562145139&single=true&output=csv"
titanic_df = pd.read_csv(url)
```

# 1. Average Age and Missing Values per Class

### Q-1: Using `groupby` make groups using the Pclass column and find out the average age and the total number of missing values in the Age column for every group.

Two statistics, one groupby. `titanic_df.groupby('Pclass')` splits the passengers into three groups (classes 1, 2, and 3). Attach the Age column and ask for two different answers.

The average is direct: `titanic_df.groupby('Pclass')['Age'].mean()` — pandas skips NaN inside each mean by default. The result is a Series: the average age of first-class passengers, then second, then third. Richer classes tended to carry older passengers.

The missing-value count needs `isnull()` inside the group: `titanic_df.groupby('Pclass')['Age'].isnull().sum()`. For each Pclass group, `isnull()` produces a boolean Series (True = Age missing), and `sum()` counts the Trues. Skipna does not help here because we are not averaging — we want the holes themselves: roughly 30, 11, and 74 in this file, a famous gap traceable to the third-class records being the least complete.

One call answers both: `titanic_df.groupby('Pclass')['Age'].agg(['mean', lambda s: s.isnull().sum()])`, but two separate expressions are clearer and teach the two idioms separately.

> Hint: `groupby('Pclass')['Age'].mean()` and `groupby('Pclass')['Age'].isnull().sum()`. Compare the counts against a whole-column `Age.isnull().sum()` afterwards.

In [ ]:
# code here

# 2. Mode Filling within Groups

### Q-2: Using `groupby` make groups using the Pclass column and fill every group's Embarked column's missing values with the mode value of that group. After that, print every group's Embarked column's value counts in ascending order.

The classic global `fillna(mode)` fills every hole with ONE number for the whole column. This question wants a per-group mode — class 1 passengers get class 1's most common port, class 3 get class 3's — because each class really did embark from different ports.

The tidy approach keeps the groupby inside the fill:

```python
titanic_df['Embarked'] = titanic_df.groupby('Pclass')['Embarked'].transform(lambda x: x.fillna(x.mode()[0]))
```

`transform` broadcasts the fill back to the ORIGINAL index, so every row keeps its place while its own class's mode lands in the holes. The lambda receives a class's Embarked series, `x.mode()[0]` takes the single most common port (mode returns a Series, hence the `[0]`), and `fillna` stamps it. Alternatively, groupby-iterate, fill each group, and concatenate back — longer but transparent.

Proof is in the output. After the fill, `titanic_df.groupby('Pclass')['Embarked'].value_counts()` counts the port distribution inside each class; the question asks for ASCENDING order within the printed result, so append `.sort_values(ascending=True)` and the 'S' port — dominant in every class — lands at the bottom of the display.

> Hint: transform + mode()[0] inside each group; then value_counts sorted ascending.

In [ ]:
# code here

# 3. Nested Groups and Average Fare

### Q-3: Make groups based on the Embarked column. And for each Embarked group, make another group based on Pclass and find out the average fare (round off up to 2 decimal places) for each Pclass for each group of Embarked.

Two levels of grouping mean a two-level structure in the question and the answer. The direct command is a two-column groupby: `titanic_df.groupby(['Embarked', 'Pclass'])['Fare'].mean()` splits first by port (C, Q, S), then by class within each port, and averages the fares. The result is a Series with a MultiIndex — port on the outer level, class on the inner.

The framework of the sample output is a nested dictionary: `{'C': {1: 105, 2: 25, 3: 11}, 'Q': {1: 90, 2: 12, 3: 11}, 'S': {1: 70, 2: 20, 3: 15}}`. Group the values and round first with `.round(2)`, then rebuild the outer-inner structure — iterate the grouped object with `for port, rows in grouped:` and build `outer[port] = {cls: round(avg, 2) for ...}`.

Reading the numbers: first class paid the most from every port, and Cherbourg's first-class passengers paid the steepest average of all (105) — exactly the story the real Titanic data tells.

> Hint: `groupby(['Embarked','Pclass'])['Fare'].mean().round(2)`, then rebuild a dict-of-dicts either by iterating the MultiIndex or by `to_dict()` on a reset frame.

In [ ]:
# code here

# 4. The Fifa World Cup 2022 Dataset

The middle block switches to the World Cup 2022 match table — one row per team-per-match, with columns including Team, Against, Goals For and Against, Attempts, On Target, Yellow and Red Cards, Passes, Passes Completed, Attempted Line Breaks, Completed Line Breaks, Possession, and Points.

```python
import pandas as pd
fifa_df = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vT3D_x_4DS6d51LKJ7ze1sxT5WpV5uiSVOFYHLwBiGru6vFyVv5h5-83AwFjxWYiWfCDjDAaarHAV-k/pub?gid=0&single=true&output=csv")
```

The single question on this data is a GroupBy masterclass. Group by `Team`, so every team's repeated rows form one group; apply a custom normalisation across the four passing columns — Passes, Passes Completed, Attempted Line Breaks, Completed Line Breaks; and finally read off a battery of per-team statistics. It deliberately combines everything you have practised: a user-defined function, `groupby().apply()`, and per-group min/max/mean/std.

The Z-normalisation formula (standard score) is:

```
Z = (X - mu) / std
```

Each value minus the group's mean, divided by the group's standard deviation — every team's passing columns become comparable regardless of how many matches they played, because the scale is reset by the team's own distribution.

# 5. Q-4 — Z-Normalisation with apply

### Q-4: Perform `groupby` based on the Team column, then perform Z-Normalisation on top of the below columns of each group:
1. Passes
2. Passes Completed
3. Attempted Line Breaks
4. Completed Line Breaks

Make a python function named `z_normalization` which takes two arguments: `group` (every group you have created) and `cols_to_perform` (a list of columns on which to perform the Z-Normalisation). For this problem, you have to use the `apply()` method. After that, find out the below values for each group: minimum Passes, maximum Passes, minimum Yellow Cards, maximum Yellow Cards, average Yellow Cards, maximum Attempted Line Breaks, minimum Attempted Line Breaks, standard deviation of Attempted Line Breaks, and average Possession.

The function signature is dictated: `z_normalization(group, cols_to_perform)`. It receives one sub-DataFrame of a team and the list of columns; inside, it normalises each requested column and adds the result back to the group:

```python
def z_normalization(group, cols_to_perform):
    for col in cols_to_perform:
        group[col + '_Z'] = (group[col] - group[col].mean()) / group[col].std()
    return group
```

Called through apply: `fifa_df.groupby('Team').apply(z_normalization, cols_to_perform=[...])` — the apply method passes each group in turn as the first positional argument, plus the extra keyword. The output is the whole table with new `_Z` columns, one per normalised metric.

The follow-up statistics need no Z-values: answer them with a per-team aggregation. Either chain onto the groupby directly or group the already-computed frame; the standard move is a dictionary agg — mapping Passes to ['min', 'max'], Yellow Cards to ['min', 'max', 'mean'], Attempted Line Breaks to ['max', 'min', 'std'], and Possession to 'mean'. That is the same dictionary-style agg you already know.

> Hint: define the function, apply it with the list of four columns; then agg a dict of per-column statistics per team.

In [ ]:
# code here

# 6. The IPL Ball-by-Ball Dataset

The final and hardest block uses the IPL ball-by-ball delivery log — one row per ball bowled, with columns such as `match_id`, `inning`, `batting_team`, `bowling_team`, `over`, `batsman`, `non_striker`, `bowler`, `batsman_runs`, `extra_type`, and `player_out`.

The file is the familiar ball-by-ball dataset (the Google Drive link in the notebook); read it with `pd.read_csv` as in the previous tasks, giving it a short name like `ipl`.

The four questions hunt through this file for records and pairings:

- Q-5 finds the batsman with the highest score while CHASING (batting in the second innings) and the best chasing strike rate among players who faced 100+ balls.
- Q-6 pairs each bowler with the batsman they have dismissed most often, breaking ties by fewer runs conceded.
- Q-7 finds the batting pair (two batsmen at the crease together) who scored the most runs together.
- Q-8 generalises this into a full dataframe of every batting pair — Batsman1, Batsman2, Runs, Average, StrikeRate.

Each requires careful masking before any groupby: `inning == 2` for chasing, `extra_type` for which balls count as balls faced, `player_out` for dismissals, and `batsman` / `non_striker` for partnerships. Do the filtering deliberately — a wrong mask produces a correct-looking wrong number.

# 7. Q-5 — Chasing Records

### Q-5: Find the batsman in the below category:
* Highest score while chasing
* Best Strike Rate while chasing and have faced 100+ balls

> Chasing means the team batting in the second inning.

Step one is the mask. A delivery is 'while chasing' when the innings is the second: `ipl[ipl['inning'] == 2]`. Everything else is built on that filtered frame.

For the HIGHEST SCORE, think match by match. A batsman's score in ONE match is the sum of `batsman_runs` over their rows in that match. So group the chasing balls by (match_id, batsman), sum the runs, and sort descending: `ipl[ipl['inning'] == 2].groupby(['match_id', 'batsman'])['batsman_runs'].sum().sort_values(ascending=False).head(1)`. The top cell is the highest chasing score in the file.

For the BEST STRIKE RATE with 100+ balls faced: strike rate = `(runs / balls faced) * 100`. Balls faced must exclude wides — a wide does not count as a ball faced. So filter the chasing deliveries to non-wide balls, group by batsman, compute total runs and total balls per batsman, keep only those with 100+ balls, and rank by runs-per-ball. The leader is the chasing strike-rate king among heavy-handed batters.

> Hint: second-innings mask first; per-match sum for highest score; (runs / balls) * 100 on non-wide balls for strike rate.

In [ ]:
# code here

# 8. Q-6 — Most Successful Bowler vs Batsman

### Q-6: Find the most successful bowler against any batsman — the bowler who has dismissed any batsman the most number of times.

> Most successful in terms of dismissals: the bowler who dismissed a batsman most often. If any two pairs have the same number of dismissals, consider runs conceded by the bowler to that batsman — whoever conceded fewer runs is more successful.

A dismissal in this file is recorded by the `player_out` column: when a ball shows a batsman in `player_out`, that batsman was dismissed on that delivery, and the same row tells you the `bowler` who did it.

The essential pipeline: keep the balls where `player_out` is not null (a dismissal happened), then group by the pair (bowler, player_out) and count. `ipl[ipl['player_out'].notnull()].groupby(['bowler', 'player_out']).size()` returns the dismissal count for every pair; sorting descending brings the killer combination to the top.

The tiebreaker adds columns: also sum `batsman_runs` for the same pairs (runs conceded by that bowler to that batsman). Sort first by the dismissal count descending, then by runs conceded ascending — the bowler who dismisses most often, and on equal counts the one who conceded fewer runs, wins. A combined aggregate — `groupby(['bowler', 'player_out']).agg(dismissals=('player_out', 'size'), runs=('batsman_runs', 'sum'))` — keeps both columns in one table for a clean two-key sort.

> Hint: filter player_out.notnull(); groupby (bowler, player_out); sort by count desc then runs asc.

In [ ]:
# code here

# 9. Q-7 — Most Successful Batting Pair

### Q-7: Find the most successful batting pair in IPL — the batting pair who have scored the most runs while playing together.

Every delivery names both batsmen: `batsman` faces the ball and `non_striker` stands at the other end. A delivered ball therefore belongs to the partnership (batsman, non_striker) — and the runs off that ball (batsman_runs) are scored by that pair while at the crease together.

So the whole question collapses into a groupby: group the deliveries by the two batsmen and sum the runs.

```python
ipl.groupby(['batsman', 'non_striker'])['batsman_runs'].sum().sort_values(ascending=False).head(1)
```

Pandas sorts the result descending, and `.head(1)` keeps the richest partnership. Expect the classic opening partnership of the file to top it — the pair who have shared the crease most often, both batting through hundreds of deliveries. Runs are credited to whichever batsman is striking, and the partnership stays the pair.

> Hint: groupby(['batsman', 'non_striker'])['batsman_runs'].sum(), sort descending, head(1).

In [ ]:
# code here

# 10. Q-8 — All Batting Pairs DataFrame

### Q-8: Make a dataframe for all batting pairs played together:
```
Batsman1 Batsman2 Runs Avg StrikeRate
```

> Just to ease this question you can count wide-balls for strike rate.

This generalises Q-7 from the single best pair to EVERY pair, with three computed columns.

Runs: group every delivery by the pair and sum `batsman_runs` — the same groupby as Q-7, kept without the head.

Average: a partnership's average means runs per DISMISSAL of the pair — total partnership runs divided by the number of times one of the pair was dismissed. Count dismissals from balls whose `player_out` belongs to either batsman. Since the question only defines the columns, choose the natural definition and state it: average = total runs / wickets lost by the pair.

Strike Rate: `(runs / balls faced) * 100`. The permission in the prompt — 'you can count wide balls for strike rate' — was designed to keep the question simple: count every delivery where either member of the pair is the striking batsman as a ball faced, wides included, instead of stripping wides out. That turns balls faced into a straightforward `count()` of the pair's deliveries.

Assemble the three pieces into a DataFrame indexed by (Batsman1, Batsman2) — or reshape with reset_index — matching the required columns. The answer table then lets you ask the same questions Q-7 answered, but for every pair in the file.

> Hint: groupby pair; sum runs; count dismissals and deliveries per pair; Avg = Runs / Dismissals, StrikeRate = (Runs / BallsFaced) * 100.

## Summary

Task 19 pushed GroupBy to its limits across three datasets. On Titanic you grouped passengers by Pclass to average the Age column and count its missing values, filled each class's missing Embarked values with that class's own mode using transform, and stacked two grouping levels — Embarked then Pclass — into a nested-dictionary report of average fare. On the World Cup data you wrote a z_normalization function and drove it through groupby().apply(), then pulled min/max/mean/std statistics per team with a dictionary agg. Finally, on IPL ball-by-ball deliveries you filtered for second-innings chases to find the highest chasing score and best strike rate, ranked bowler-batsman pairs by dismissals with runs-conceded tiebreaks, identified the highest-scoring partnership, and built a complete batting-pair dataframe of Runs, Average, and StrikeRate. The recurring lesson: filter rigorously first, then group — every hard question here was a clean mask plus a groupby away.

In [ ]:
# code here